In [1]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [2]:
# Read Data

df = pd.read_csv("/content/data/domain_specific_chatbot_data.csv")

print(df.head())

                                               query  \
0  What are the side effects of the COVID-19 vacc...   
1  How can I schedule an appointment with my doctor?   
2  What should I do if I miss a dose of my medica...   
3                How can I check my account balance?   
4     What is the interest rate for a personal loan?   

                                            response                intent  \
0  Common side effects of the COVID-19 vaccine in...  side effects inquiry   
1  You can schedule an appointment by calling our...   appointment booking   
2  If you miss a dose, take it as soon as you rem...    medication inquiry   
3  You can check your balance by logging into you...       balance inquiry   
4  The current interest rate for personal loans i...          loan inquiry   

       domain  
0  healthcare  
1  healthcare  
2  healthcare  
3     finance  
4     finance  


In [3]:
# Explore Data

print(f"Question: {df["query"][0]}")
print(f"Answer: {df["response"][0]}")

print(f"Data Shape: {df.shape}")

Question: What are the side effects of the COVID-19 vaccine?
Answer: Common side effects of the COVID-19 vaccine include soreness at the injection site, fever, and fatigue.
Data Shape: (3000, 4)


In [4]:
# Split Data

from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Train Shape: {train_df.shape}")
print(f"Test Shape: {val_df.shape}")

Train Shape: (2400, 4)
Test Shape: (600, 4)


In [5]:
# Reset Data Index

print(f"Train Data\n: {train_df}")
print(f"Val Data\n: {val_df}")

train_data = train_df.reset_index(drop=True)
val_data = val_df.reset_index(drop=True)

print(f"Train Data\n: {train_data}")
print(f"Val Data\n: {val_data}")

Train Data
:                                                   query  \
642   What should I do if I miss a dose of my medica...   
700   What are the side effects of the COVID-19 vacc...   
226                       What are the symptoms of flu?   
1697  How do I update my contact details on my account?   
1010  What are the side effects of the COVID-19 vacc...   
...                                                 ...   
1638  Can I make changes to my loan repayment schedule?   
1095           I lost my credit card, what should I do?   
1130  What are the side effects of the COVID-19 vacc...   
1294     What is the interest rate for a personal loan?   
860   What are the side effects of the COVID-19 vacc...   

                                               response  \
642   If you miss a dose, take it as soon as you rem...   
700   Common side effects of the COVID-19 vaccine in...   
226   Flu symptoms include fever, cough, sore throat...   
1697  To update your contact details, log 

In [6]:
# Clean Text

import re

def clean_text(text):
  text = re.sub(r"\r\n", " ", text)
  text = re.sub(r"\s+", " ", text)
  text = re.sub(r"<.*?>", "", text)
  text = text.strip().lower()

  return text

train_data["query"] = train_data["query"].apply(clean_text)
train_data["response"] = train_data["response"].apply(clean_text)

val_data["query"] = val_data["query"].apply(clean_text)
val_data["response"] = val_data["response"].apply(clean_text)

print(train_data.head())

                                               query  \
0  what should i do if i miss a dose of my medica...   
1  what are the side effects of the covid-19 vacc...   
2                      what are the symptoms of flu?   
3  how do i update my contact details on my account?   
4  what are the side effects of the covid-19 vacc...   

                                            response                intent  \
0  if you miss a dose, take it as soon as you rem...    medication inquiry   
1  common side effects of the covid-19 vaccine in...  side effects inquiry   
2  flu symptoms include fever, cough, sore throat...  flu symptoms inquiry   
3  to update your contact details, log into your ...        contact update   
4  common side effects of the covid-19 vaccine in...  side effects inquiry   

       domain  
0  healthcare  
1  healthcare  
2  healthcare  
3     finance  
4  healthcare  


In [7]:
# Load Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
# Apply Tokenization

def preprocess_function(examples):
  inputs = tokenizer(examples["query"], max_length = 250, padding = "max_length", truncation = True)
  targets = tokenizer(examples["response"], max_length = 250, padding = "max_length", truncation = True)

  inputs["labels"] = targets["input_ids"]

  return inputs

train_dataset = train_data.apply(preprocess_function, axis = 1)
val_dataset = val_data.apply(preprocess_function, axis = 1)

In [9]:
# Show Data

print("Before Tokenization")
print(f"Question: {train_data["query"][0]}")
print(f"Answer: {train_data["response"][0]}")

print("===================")

print("After Tokenization")
print(f"Question: {train_dataset[0]["input_ids"]}")
print(f"Answer: {train_dataset[0]["labels"]}")

Before Tokenization
Question: what should i do if i miss a dose of my medication?
Answer: if you miss a dose, take it as soon as you remember unless it's almost time for your next dose. if you’re unsure, contact your healthcare provider.
After Tokenization
Question: [125, 225, 3, 23, 103, 3, 99, 3, 23, 3041, 3, 9, 6742, 13, 82, 7757, 58, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [10]:
# Set Model

model = T5ForConditionalGeneration.from_pretrained("t5-base")

training_args = TrainingArguments(
  output_dir = "../results",
  num_train_epochs = 6,
  per_device_train_batch_size = 8,
  per_device_eval_batch_size = 8,
  warmup_steps = 500,
  weight_decay = 0.01,
  logging_dir = "../logs",
  logging_steps = 50,
  save_steps = 500,
  eval_steps = 50,
  do_eval = True,
)

trainer = Trainer(
  model = model,
  args = training_args,
  train_dataset = train_dataset,
  eval_dataset = val_dataset,
)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [11]:
# Train Model

trainer.train()

Step,Training Loss
50,13.242239
100,7.425972
150,1.439551
200,0.181521
250,0.069589
300,0.028577
350,0.013985
400,0.007222
450,0.004557
500,0.003295


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1800, training_loss=0.6231159941738265, metrics={'train_runtime': 2022.0358, 'train_samples_per_second': 7.122, 'train_steps_per_second': 0.89, 'total_flos': 4281735168000000.0, 'train_loss': 0.6231159941738265, 'epoch': 6.0})

In [12]:
# Save Models

model.save_pretrained("../saved_files/chatbot")
tokenizer.save_pretrained("../saved_files/tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../saved_files/tokenizer/tokenizer_config.json',
 '../saved_files/tokenizer/tokenizer.json')

In [13]:
# Use Models

model = T5ForConditionalGeneration.from_pretrained("../saved_files/chatbot")
tokenizer = T5Tokenizer.from_pretrained("../saved_files/tokenizer")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

In [14]:
# Build Chatbot

device = model.device

def chatbot(query):
  query = clean_text(query)
  # tokenizer return: input_ids, attention_mask
  # tokenizer.encode return: input_ids
  input_ids = tokenizer(query, return_tensors = "pt", max_length = 250, truncation = True)

  inputs = {key: value.to(device) for key, value in input_ids.items()}

  outputs = model.generate(
    input_ids["input_ids"],
    max_length = 250,
    num_beams = 5,
    early_stopping = True,
  )

  return tokenizer.decode(outputs[0], skip_special_tokens = True)


while True:
  user_input = input("You: ")
  if user_input.lower() == "exit":
    break
  response = chatbot(user_input)
  print(f"Chatbot: {response}")

You: How to login
Chatbot: login to log into your account and go to the main page.
You: How to schedule an appointement?
Chatbot: you can schedule an appointment by calling our office or using our online portal.
You: exit


In [19]:
# Archive Model

import shutil

shutil.make_archive("chatbot_model", "zip", "../saved_files")

from google.colab import files

files.download("chatbot_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>